In [2]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

from xgboost import XGBClassifier

In [6]:
df = pd.read_csv("train_test_network.csv")
df.head()

,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,-,290.371539,101568,2592,OTH,...,0,0,-,-,-,-,-,-,1,backdoor
1,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000102,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
2,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000148,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
3,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000113,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
4,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000130,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor


In [7]:
df = df.sample(30000, random_state=42)

In [8]:
X = df.drop(["label","type"],axis=1)
y = df["label"]

In [9]:
le = LabelEncoder()
for col in X.columns:
    if X[col].dtype=="object":
        X[col]=le.fit_transform(X[col].astype(str))

In [10]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.30,random_state=42)

In [11]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [14]:
xgb = XGBClassifier(n_estimators=200,max_depth=8,learning_rate=0.1,subsample=0.8,colsample_bytree=0.8,random_state=42)

In [15]:
# start=time.time()
xgb.fit(X_train,y_train)
# end=time.time()
# print("Training Time:",end-start)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [16]:
from sklearn.ensemble import IsolationForest
X_normal = X_train[y_train==0]
iso = IsolationForest(n_estimators=200,contamination=0.30,random_state=42)
iso.fit(X_normal)

IsolationForest(contamination=0.3, n_estimators=200, random_state=42)

In [18]:
import numpy as np
xgb_preds = xgb.predict(X_test)
iso_preds = iso.predict(X_test)
iso_preds=np.where(iso_preds==-1,1,0)
hybrid_preds=[]
for x,i in zip(xgb_preds,iso_preds):
    # if XGBoost says attack
    if x==1:
        hybrid_preds.append(1)
    # if XGBoost misses but anomaly detector flags suspicious
    elif i==1:
        hybrid_preds.append(1)
    else:
        hybrid_preds.append(0)
hybrid_preds=np.array(hybrid_preds)

In [19]:
from sklearn.metrics import accuracy_score,classification_report
print("Hybrid IDS Accuracy:",accuracy_score(y_test,hybrid_preds))
print(classification_report(y_test,hybrid_preds))

Hybrid IDS Accuracy: 0.9273333333333333
              precision    recall  f1-score   support

           0       1.00      0.69      0.82      2106
           1       0.91      1.00      0.95      6894

    accuracy                           0.93      9000
   macro avg       0.96      0.84      0.89      9000
weighted avg       0.93      0.93      0.92      9000



In [21]:
import time
from sklearn.metrics import accuracy_score,classification_report


# --------------------------
# TRAINING TIME
# --------------------------

start_train=time.time()

xgb.fit(
X_train,
y_train
)

end_train=time.time()

print(
"XGBoost Training Time:",
end_train-start_train
)


# --------------------------
# PREDICTION TIME
# --------------------------

start_pred=time.time()

xgb_preds=xgb.predict(
X_test
)

end_pred=time.time()

print(
"XGBoost Prediction Time:",
end_pred-start_pred
)


# --------------------------
# Accuracy
# --------------------------

print(
"XGBoost Accuracy:",
accuracy_score(
y_test,
xgb_preds
)
)

print(
classification_report(
y_test,
xgb_preds
))

XGBoost Training Time: 0.44170165061950684
XGBoost Prediction Time: 0.009283304214477539
XGBoost Accuracy: 0.9997777777777778
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2106
           1       1.00      1.00      1.00      6894

    accuracy                           1.00      9000
   macro avg       1.00      1.00      1.00      9000
weighted avg       1.00      1.00      1.00      9000



In [20]:
import time
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score,classification_report


# --------------------------
# TRAINING TIME START
# --------------------------

start_train=time.time()


# Train XGBoost
xgb.fit(
X_train,
y_train
)


# Train Isolation Forest
X_normal=X_train[y_train==0]

iso=IsolationForest(
n_estimators=200,
contamination=0.30,
random_state=42
)

iso.fit(X_normal)


end_train=time.time()


print(
"Hybrid Training Time:",
end_train-start_train
)


# --------------------------
# PREDICTION TIME START
# --------------------------

start_pred=time.time()


# Stage 1 predictions
xgb_preds=xgb.predict(X_test)


# Stage 2 predictions
iso_preds=iso.predict(X_test)

iso_preds=np.where(
iso_preds==-1,
1,
0
)


# Hybrid fusion logic
hybrid_preds=[]

for x,i in zip(
xgb_preds,
iso_preds
):

    if x==1 or i==1:
        hybrid_preds.append(1)

    else:
        hybrid_preds.append(0)


hybrid_preds=np.array(
hybrid_preds
)


end_pred=time.time()


print(
"Hybrid Prediction Time:",
end_pred-start_pred
)


# --------------------------
# Accuracy
# --------------------------

print(
"Hybrid Accuracy:",
accuracy_score(
y_test,
hybrid_preds
)
)

print(
classification_report(
y_test,
hybrid_preds
))

Hybrid Training Time: 0.9943797588348389
Hybrid Prediction Time: 0.2013263702392578
Hybrid Accuracy: 0.9273333333333333
              precision    recall  f1-score   support

           0       1.00      0.69      0.82      2106
           1       0.91      1.00      0.95      6894

    accuracy                           0.93      9000
   macro avg       0.96      0.84      0.89      9000
weighted avg       0.93      0.93      0.92      9000

